### SVI Dataset

In [1]:
import numpy as np
import pandas as pd

#Change city
city = 'Texas'

#Social vulnerability features
SV = ['EPL_POV150','EPL_UNEMP','EPL_HBURD','EPL_NOHSDP','EPL_UNINSUR','RPL_THEME1',
      'EPL_AGE65','EPL_AGE17','EPL_DISABL','EPL_SNGPNT','EPL_LIMENG','RPL_THEME2',
      'EPL_MINRTY',
      'EPL_MUNIT','EPL_MOBILE','EPL_CROWD','EPL_NOVEH','EPL_GROUPQ','RPL_THEME4']

#Tract-Zip dataset processing
tract_zip = pd.read_excel(city + '_TRACT_ZIP.xlsx','Sheet1')
tract_zip.sort_values(by=['TRACT','RES_RATIO'], inplace=True)
tract_zip = tract_zip.reset_index(drop=True)
TRACT_df = tract_zip.groupby('TRACT').agg({'ZIP':'last','RES_RATIO':'max'}).reset_index()

tract_zip_dic = dict(zip(TRACT_df['TRACT'], TRACT_df['ZIP']))
tract_pop_dic = dict(zip(TRACT_df['TRACT'], TRACT_df['RES_RATIO']))

#Tract-SVI dataset processing
SVI_df = pd.read_csv(city + '_TRACT_SVI.csv')
SVI_df = SVI_df[['FIPS','E_TOTPOP'] + SV]

SVI_df['ZIP_CODE'] = -999
for key,value in tract_zip_dic.items():
    if key in list(SVI_df['FIPS']):
        SVI_df.loc[SVI_df[SVI_df['FIPS'] == key].index.tolist()[0],'ZIP_CODE'] = value

SVI_df['RES_RATIO'] = -999
for key,value in tract_pop_dic.items():
    if key in list(SVI_df['FIPS']) and value > 0.0:
        SVI_df.loc[SVI_df[SVI_df['FIPS'] == key].index.tolist()[0],'RES_RATIO'] = value

SVI_df = SVI_df.replace(-999, np.nan)
SVI_df = SVI_df.dropna()

SVI_df['POP_RATIO'] = SVI_df['E_TOTPOP'] * SVI_df['RES_RATIO']

#Zip codes integration
zip_codes = list(set(SVI_df['ZIP_CODE']))
zip_codes.sort()
SVI_dic = {'ZIP_CODE':zip_codes}
for i in range(len(SV)):
    SVI = []
    for code in zip_codes:
        df = SVI_df[SVI_df['ZIP_CODE'].isin([code])].reset_index()
        df['Weight_SV'] = df[SV[i]] * df['POP_RATIO']
        SVI.append(round(df['Weight_SV'].sum() / df['POP_RATIO'].sum(), 4))
    SVI_dic[SV[i]] = SVI

#Data output
ZIP_SVI = pd.DataFrame(SVI_dic)
ZIP_SVI.to_csv('SVI_Dataset.csv',index=False)